# Focus Score Estimates

Loads per-image focus scores for all spher52 plates, joins with metadata, and reports mean focus scores per compound per Z-slice.

In [ ]:
# --- repo path bootstrap (added by the port) ---
import sys, pathlib
ROOT = next(p for p in pathlib.Path.cwd().parents if (p / "utils" / "paths.py").is_file())
sys.path.insert(0, str(ROOT))
from utils.paths import (profiles, features, feature_output, figdir, metadata,
                         data_dir, external, require)
from utils.panels import save_panel

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path

In [ ]:
# Segmentation focus scores, shipped with the repo (18 small CSVs)
BASE = ROOT / "analysis" / "3_SupplFigure2" / "data" / "spher-colo52"
META_PATH = Path(metadata("spher_colo52-metadata.csv", "exp1_main"))
OUT_DIR = figdir("SupplFig2")

# Load and tag each plate's focus_scores.csv
frames = []
for plate_dir in sorted(BASE.glob("PB*")):
    csv = plate_dir / "segmentation_output" / "focus_scores.csv"
    if not csv.exists():
        print(f"Missing (not yet run?): {csv}")
        continue
    barcode = plate_dir.name.split("_")[0]
    frames.append(pd.read_csv(csv).assign(barcode=barcode))

focus = pd.concat(frames, ignore_index=True)
print(f"Loaded {len(focus)} rows across {focus['barcode'].nunique()} plates")
focus.head()

In [ ]:
# Load metadata — keep compound name, code, cell line
meta = pd.read_csv(META_PATH)[["barcode", "well_id", "cmpd_code", "cmpdname", "cell_line", "pert_type"]]

# Compounds of interest (cross-referenced from metadata)
KEEP_CODES = {"dmso", "colo-040", "colo-041", "colo-044"}  # DMSO, SN-38, Binimetinib, abemaciclib

# Join and filter to compounds of interest
joined = (
    focus
    .merge(meta, left_on=["barcode", "well"], right_on=["barcode", "well_id"], how="left")
    .query("cmpd_code in @KEEP_CODES")
    .reset_index(drop=True)
)

n_unmatched = focus.merge(meta, left_on=["barcode", "well"], right_on=["barcode", "well_id"], how="left")["cmpd_code"].isna().sum()
if n_unmatched:
    print(f"Warning: {n_unmatched} rows had no metadata match (dropped)")

print(f"Joined rows after filter: {len(joined)}")
print(joined.groupby(["cmpd_code", "cmpdname"]).size().reset_index(name="count"))
joined.head()

In [ ]:
# Mean focus scores per compound × z-slice × channel
mean_focus = (
    joined
    .groupby(["cmpd_code", "cmpdname", "z", "channel"])
    .agg(
        mean_laplacian_var=("focus_laplacian_var", "mean"),
        std_laplacian_var=("focus_laplacian_var", "std"),
        mean_normalized_var=("focus_normalized_var", "mean"),
        std_normalized_var=("focus_normalized_var", "std"),
        n_wells=("focus_laplacian_var", "count"),
    )
    .reset_index()
    .sort_values(["cmpd_code", "channel", "z"])
)

print(f"Unique compounds: {mean_focus['cmpd_code'].nunique()}")
print(f"Unique channels:  {mean_focus['channel'].nunique()}")
print(f"Z range:          {mean_focus['z'].min()} – {mean_focus['z'].max()}")
mean_focus.head(10)

In [ ]:
# Save the per-compound per-slice table
out_path = OUT_DIR / "focus_mean_per_compound_slice.csv"
mean_focus.to_csv(out_path, index=False)
print(f"Saved: {out_path}")

In [ ]:
COMPOUND_COLORS = {
    "dmso":     "#AAAAAA",  # light gray
    "colo-040": "#111111",  # near black  — SN-38
    "colo-041": "#555555",  # dark gray   — Binimetinib
    "colo-044": "#888888",  # medium gray — abemaciclib
}

COMPOUND_LABELS = {
    "dmso":     "DMSO",
    "colo-040": "SN-38",
    "colo-041": "Binimetinib",
    "colo-044": "abemaciclib",
}

channels = sorted(mean_focus["channel"].unique().tolist())
n_ch = len(channels)

fig, axes = plt.subplots(1, n_ch, figsize=(4 * n_ch, 5), sharey=False)
if n_ch == 1:
    axes = [axes]

for ax, ch in zip(axes, channels):
    ch_df = mean_focus[mean_focus["channel"] == ch]

    for code, color in COMPOUND_COLORS.items():
        cmpd_df = ch_df[ch_df["cmpd_code"] == code].sort_values("z")
        if cmpd_df.empty:
            continue

        zs   = cmpd_df["z"].to_numpy()
        mean = cmpd_df["mean_laplacian_var"].to_numpy()
        std  = cmpd_df["std_laplacian_var"].fillna(0).to_numpy()

        ax.plot(zs, mean, color=color, linewidth=2, label=COMPOUND_LABELS[code], zorder=3)
        ax.fill_between(zs, mean - std, mean + std, color=color, alpha=0.18, zorder=2)

    ax.set_title(ch)
    ax.set_xlabel("Z slice")
    ax.set_ylabel("Mean Laplacian variance")
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.grid(axis="y", linestyle="--", alpha=0.35)
    ax.legend(fontsize=8)

fig.suptitle("Mean focus (Laplacian variance) per compound per Z-slice\n(shading = ±1 SD)", y=1.02)
plt.tight_layout()
# Set up the plotting parameters
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

save_panel(fig, "SupplFig2c", data=mean_focus,
           caption="Mean focus (Laplacian variance) per compound and z-slice",
           notebook="analysis/3_SupplFigure2/focus_estimates.ipynb")
print("Saved: focus_by_compound_z.png + .pdf")
plt.show()

In [ ]:
channels = sorted(mean_focus["channel"].unique().tolist())
n_ch = len(channels)

fig3, axes3 = plt.subplots(1, n_ch, figsize=(4 * n_ch, 5), sharey=False)
if n_ch == 1:
    axes3 = [axes3]

for ax, ch in zip(axes3, channels):
    ch_df = mean_focus[mean_focus["channel"] == ch]

    for code, color in COMPOUND_COLORS.items():
        cmpd_df = ch_df[ch_df["cmpd_code"] == code].sort_values("z")
        if cmpd_df.empty:
            continue

        zs   = cmpd_df["z"].to_numpy()
        mean = cmpd_df["mean_normalized_var"].to_numpy()
        std  = cmpd_df["std_normalized_var"].fillna(0).to_numpy()

        ax.plot(zs, mean, color=color, linewidth=2, label=COMPOUND_LABELS[code], zorder=3)
        ax.fill_between(zs, mean - std, mean + std, color=color, alpha=0.18, zorder=2)

    ax.set_title(ch)
    ax.set_xlabel("Z slice")
    ax.set_ylabel("Mean normalized variance (var/mean)")
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.grid(axis="y", linestyle="--", alpha=0.35)
    ax.legend(fontsize=8)

fig3.suptitle("Mean focus (normalized variance) per compound per Z-slice\n(shading = ±1 SD)", y=1.02)
plt.tight_layout()
# Set up the plotting parameters
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

# [not a paper panel] fig3.savefig(OUT_DIR / "focus_normalized_var_by_compound_z.png", dpi=150, bbox_inches="tight")
# [not a paper panel] fig3.savefig(OUT_DIR / "focus_normalized_var_by_compound_z.pdf", bbox_inches="tight")
print("Saved: focus_normalized_var_by_compound_z.png + .pdf")
plt.show()

In [ ]:
# Mean normalized variance per compound × cell_line × z × channel
mean_focus_cl = (
    joined
    .groupby(["cmpd_code", "cmpdname", "cell_line", "z", "channel"])
    .agg(
        mean_normalized_var=("focus_normalized_var", "mean"),
        std_normalized_var=("focus_normalized_var", "std"),
        n_wells=("focus_normalized_var", "count"),
    )
    .reset_index()
    .sort_values(["cmpd_code", "cell_line", "channel", "z"])
)

cell_lines = sorted(mean_focus_cl["cell_line"].unique().tolist())
channels   = sorted(mean_focus_cl["channel"].unique().tolist())
n_cl = len(cell_lines)
n_ch = len(channels)

fig4, axes4 = plt.subplots(n_cl, n_ch, figsize=(4 * n_ch, 4 * n_cl), sharey=False, sharex=True)
if n_cl == 1:
    axes4 = [axes4]
if n_ch == 1:
    axes4 = [[row] for row in axes4]

for row, cl in enumerate(cell_lines):
    cl_df = mean_focus_cl[mean_focus_cl["cell_line"] == cl]
    for col, ch in enumerate(channels):
        ax = axes4[row][col]
        ch_df = cl_df[cl_df["channel"] == ch]

        for code, color in COMPOUND_COLORS.items():
            cmpd_df = ch_df[ch_df["cmpd_code"] == code].sort_values("z")
            if cmpd_df.empty:
                continue

            zs   = cmpd_df["z"].to_numpy()
            mean = cmpd_df["mean_normalized_var"].to_numpy()
            std  = cmpd_df["std_normalized_var"].fillna(0).to_numpy()

            ax.plot(zs, mean, color=color, linewidth=2, label=COMPOUND_LABELS[code], zorder=3)
            ax.fill_between(zs, mean - std, mean + std, color=color, alpha=0.18, zorder=2)

        if row == 0:
            ax.set_title(ch)
        if col == 0:
            ax.set_ylabel(f"{cl}\nnorm. variance (var/mean)")
        ax.set_xlabel("Z slice")
        ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
        ax.grid(axis="y", linestyle="--", alpha=0.35)
        ax.legend(fontsize=7)

fig4.suptitle("Normalized variance per cell line per Z-slice\n(shading = ±1 SD)", y=1.02)
plt.tight_layout()

# [not a paper panel] fig4.savefig(OUT_DIR / "focus_normalized_var_by_cellline_z.png", dpi=150, bbox_inches="tight")
# [not a paper panel] fig4.savefig(OUT_DIR / "focus_normalized_var_by_cellline_z.pdf", bbox_inches="tight")
print("Saved: focus_normalized_var_by_cellline_z.png + .pdf")
plt.show()